In [1]:
import sys, json, warnings
from pathlib import Path
import pandas as pd, numpy as np

warnings.filterwarnings("ignore")
sys.path.append(str(Path.cwd().parent / "src"))
from config import DATA_PROCESSED, RESULTS_TABLES, RANDOM_SEED
from features import (
    add_lexical_features,
    add_evidentiary_features,
    add_sentiment_features,
    add_time_of_day,
)

np.random.seed(RANDOM_SEED)
print("Ready")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 27776.85it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5700.77it/s]

Ready


In [2]:
reddit  = pd.read_parquet(DATA_PROCESSED / "reddit_processed.parquet")
twitter = pd.read_parquet(DATA_PROCESSED / "twitter_processed.parquet")
print("Reddit:", reddit.shape, " Twitter:", twitter.shape)

Reddit: (9406, 35)  Twitter: (11729, 27)


In [3]:
reddit  = add_lexical_features(reddit,  "clean_text")
twitter = add_lexical_features(twitter, "clean_text")
print("Lexical done")

Lexical done


In [4]:
reddit  = add_evidentiary_features(reddit,  "clean_text")
twitter = add_evidentiary_features(twitter, "clean_text")
print("Evidentiary done")

Evidentiary done


In [5]:
# If your dataset is very large, sample first for testing:
# reddit  = reddit.sample(5000, random_state=RANDOM_SEED)
# twitter = twitter.sample(5000, random_state=RANDOM_SEED)

reddit  = add_sentiment_features(reddit,  "clean_text")
twitter = add_sentiment_features(twitter, "clean_text")
print("Sentiment done")

Sentiment done


In [6]:
reddit  = add_time_of_day(reddit)
twitter = add_time_of_day(twitter)
print("Temporal done")

Temporal done


In [7]:
reddit["platform"]  = "Reddit"
twitter["platform"] = "X"

reddit.to_parquet(DATA_PROCESSED / "reddit_with_features.parquet", index=False)
twitter.to_parquet(DATA_PROCESSED / "twitter_with_features.parquet", index=False)

combined = pd.concat([reddit, twitter], ignore_index=True)
combined.to_parquet(DATA_PROCESSED / "combined_with_features.parquet", index=False)

print("Saved. Reddit:", len(reddit), "Twitter:", len(twitter), "Combined:", len(combined))

Saved. Reddit: 9406 Twitter: 11729 Combined: 21135


In [8]:
key = ["fk_grade", "ttr", "rigor_score", "sentiment_score",
       "evidentiary_count", "link_count", "engagement_normalized"]
print("Reddit means:\n",  reddit[key].mean(numeric_only=True).round(4))
print("\nX means:\n",     twitter[key].mean(numeric_only=True).round(4))

Reddit means:
 fk_grade                 8.4687
ttr                      0.8637
rigor_score              0.0000
sentiment_score          0.0000
evidentiary_count        0.9346
link_count               0.0000
engagement_normalized    0.2249
dtype: float64

X means:
 fk_grade                 10.7959
ttr                       0.8813
rigor_score               0.0000
sentiment_score           0.0000
evidentiary_count         0.5405
link_count                0.0000
engagement_normalized     0.4185
dtype: float64
